## Install Libraries

Runs candidate retrieval (architecture §5.2/§5.3) from the feedback that
`GRF/feedback.ipynb` already generated: load that bundle, load the KB data
(`kb.csv`) and the single-file KB artifact built by `build_KB.ipynb`, load the
gold BEL data for evaluation, run the retrieval pipeline once per scenario, and
save one candidates file per scenario.

**No language model is loaded here.** The feedback text is read from the
bundle, so this notebook only needs the encoder and the vector index — that is
the point of the split: the 32B feedback model's weights and the KB vectors
never compete for the same GPU.

This notebook reports **acc@k** (candidate retrieval). The end-to-end
re-ranking/accuracy stage is separate: `reranking.ipynb`.

Sources: `docs/architecture/README.md`, `paper/reference_abstract.md`,
`source/GRF/README.md`.

Same encoder pin as `source/GRF/requirements.txt`. `transformers` is **not**
installed: nothing in this notebook decodes text. `tqdm` is installed
explicitly because every long stage reports progress.


In [18]:
%pip install "tqdm>=4.66" "sentence-transformers>=2.7" "pycauset>=0.6.1" "cupy-cuda12x[ctk]"

## Import Libraries

Third-party imports only. The `common` and `GRF` packages live inside the
repository clone, so project modules are imported at the end of the
repository-setup stage below.


In [19]:
from __future__ import annotations

import gc
import json
import os
import subprocess
import sys
import time
from contextlib import contextmanager
from pathlib import Path

from tqdm.auto import tqdm

# Hugging Face auth for gated checkpoints; the token comes from the HF_TOKEN env var.
from huggingface_hub import login as _hf_login
if os.environ.get("HF_TOKEN"):
    _hf_login(token=os.environ["HF_TOKEN"])

print("Python:", sys.version.split()[0])
print("tqdm:", __import__("tqdm").__version__)


@contextmanager
def progress_bar(total, desc: str, unit: str = "item"):
    """Drive a tqdm bar from a progress_callback(done, total)."""
    bar = tqdm(total=total, desc=desc, unit=unit)

    def report(done: int, total=None) -> None:
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        bar.close()

@contextmanager
def phase_progress(prefix: str = ""):
    """One tqdm bar per pipeline phase, created when that phase first reports.

    ``link_batch`` reports ``(phase, done, total)`` once per phase and the
    phases run in order, so each gets its own bar instead of one bar counting
    a mixture of work. There is no feedback phase here: the text comes from
    the bundle `feedback.ipynb` wrote. A phase whose ``total`` is None (the
    encoder batches internally) shows a running count, not a percentage.
    ``prefix`` names the scenario, since one notebook run walks several.
    """
    bars = {}

    def report(phase, done, total=None) -> None:
        bar = bars.get(phase)
        if bar is None:
            bar = tqdm(total=total, desc=f"{prefix}{phase}", unit="item", leave=True)
            bars[phase] = bar
        if total is not None and total != bar.total:
            bar.total = total
        bar.update(done - bar.n)

    try:
        yield report
    finally:
        for bar in bars.values():
            bar.close()


Python: 3.13.16
tqdm: 4.67.3


## Constants and Configuration

All paths and retrieval parameters live here. The Git repository and branch are
preconfigured.

Drive layout:

- `COMMON_DIR/` — shared root (`DRIVE_ROOT/work/nlp/UMLS/entity_linking`),
- `COMMON_DIR/data/` — shared inputs: `kb.csv`, the `.npz` artifact from
  `build_KB.ipynb`, and the gold BEL JSONL,
- `COMMON_DIR/output/{METHOD}/` — this method's outputs (`METHOD = "grf"`).

`FEEDBACK_BUNDLE_PATH` is the file `GRF/feedback.ipynb` wrote. Its namespace is
`<model>|<prompt digest>`, and this notebook passes the same
`feedback_namespace(LLM_MODEL)`; a bundle generated by another model or against
another `prompts.yaml` is refused instead of being replayed.

`FEEDBACK_SCENARIOS` lists the feedback subsets to evaluate and `FUSIONS` the
two fusion pipelines; one run walks every pair, so the paper's matrix comes out
of a single execution. Each subset is selected from the bundle by each record's
`kind` key: the bundle may hold all three kinds, and the single-type and
All-in-One rows replay the same file. A requested kind the bundle does not
contain fails loudly in stage 6. `N_SYN` and `TARGET_KB` are recorded for
provenance only: they shaped the prompts that already ran. Retrieval stops at
the ranking: no re-ranking runs in this notebook. `rerank="top1"` is the
paper's *Pick Top-1 (không re-rank)* branch (Bảng 3) and only fills
`Response.cui` with the ranking's first entry; `reranking.ipynb` applies the
real re-ranker to the saved candidates.


In [20]:
# ---------------------------------------------------------------------------
# Git repository (REQUIRED: replace the placeholder with your repository URL)
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"                                             # TODO
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive layout
#   COMMON_DIR/data/                shared inputs (kb.csv, gold data, feedback)
#   COMMON_DIR/output/{ENCODER_NAME}/{TRANSLATE_NAME}/ranking/grf/  ranking outputs (shared)
# ---------------------------------------------------------------------------
METHOD = "grf"
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"
COMMON_DIR = DRIVE_ROOT / "work" / "nlp" / "UMLS" / "entity_linking"
DRIVE_DATA_DIR = COMMON_DIR / "data"
OUTPUT_ROOT = COMMON_DIR / "output"

# ---------------------------------------------------------------------------
# Encoder (must match the encoder_info stored in the KB artifact)
# ---------------------------------------------------------------------------
ENCODER_MODEL = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
ENCODER_NAME = ENCODER_MODEL.split("/")[-1]
TRANSLATE_MODEL = "Qwen/Qwen3-32B"   # must match translate_gold.ipynb MODEL_ID
TRANSLATE_NAME = TRANSLATE_MODEL.split("/")[-1]
ENCODER_BATCH_SIZE = 32
DEVICE = "cuda"                # "cuda" (Colab GPU) or "cpu"

RANKING_DIR = OUTPUT_ROOT / ENCODER_NAME / TRANSLATE_NAME / "ranking" / METHOD
KB_CSV_PATH = DRIVE_DATA_DIR / "kb.csv"                         # KB data (input)
KB_ARTIFACT_PATH = DRIVE_DATA_DIR / TRANSLATE_NAME / ENCODER_NAME / "grf_kb.npz"  # from build_KB.ipynb
GOLD_DATA_PATH = DRIVE_DATA_DIR / "translated.jsonl"            # gold BEL JSONL
FEEDBACK_BUNDLE_PATH = RANKING_DIR / "grf_feedback_bundle.jsonl"  # from GRF/feedback.ipynb
CANDIDATES_DIR = RANKING_DIR / "candidates"                     # one JSONL per scenario
SUMMARY_PATH = RANKING_DIR / "grf_retrieval_summary.json"       # acc@k per scenario

# ---------------------------------------------------------------------------
# Retrieval scenarios (architecture 9.1). One notebook run evaluates every
# pair below and writes one candidates file per pair. The bundle may hold every
# kind; each scenario selects its subset by the record's `kind`.
# ---------------------------------------------------------------------------
FEEDBACK_SCENARIOS = (
    ("none", ()),                        # no GRF: the reference row
    ("def", ("def",)),                   # single entity definition
    ("syn", ("syn",)),                   # single N synonyms
    ("std", ("std",)),                   # single standard name
    ("all", ("def", "syn", "std")),      # All-in-One
)
FUSIONS = ("vector", "rrf")     # Pipeline 1 and Pipeline 2

N_SYN = 5                       # provenance only: it shaped the saved prompts
TARGET_KB = "KB"                # provenance only: it filled the `std` instruction
TOP_K_RETRIEVAL = 10
TOP_K_CONCEPT = 5               # 1..10; candidates saved for e2e
USE_CONTEXT_QUERY = False        # True = mention + context; False = encode(mention) alone
INDEX_KB = None                 # optional source_kb filter; None = all entries

# The bundle's namespace is "<model>|<prompt digest>". Only the model name is
# needed here; the digest is read from the prompts.yaml in this clone, so an
# edited template invalidates the old bundle rather than reusing it.
LLM_MODEL = "Qwen/Qwen3-32B"    # the model that generated the bundle

# ---------------------------------------------------------------------------
# Evaluation / runtime
# ---------------------------------------------------------------------------
MAX_SAMPLES = None              # None = all messages; an int truncates for smoke runs
ACC_K = (1, 5)                  # acc@k columns (clipped to TOP_K_CONCEPT)
CACHE_DIR = Path("/content/grf_cache")
HF_HOME = CACHE_DIR / "huggingface"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_HOME))

_START = time.time()


def log(message: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} +{time.time() - _START:6.1f}s] {message}", flush=True)


def require_configured(value, name: str):
    text = str(value)
    if "REPLACE_ME" in text:
        raise ValueError(f"{name} is still a placeholder: {text!r}. Set it in the constants cell.")
    return value

log("Configuration loaded.")
print(f"Repository   : {GIT_REPO_URL} ({GIT_BRANCH})")
print(f"KB data      : {KB_CSV_PATH}")
print(f"KB artifact  : {KB_ARTIFACT_PATH}")
print(f"Gold data    : {GOLD_DATA_PATH}")
print(f"Ranking dir  : {RANKING_DIR}")
print(f"Feedback     : {FEEDBACK_BUNDLE_PATH} (from feedback.ipynb, model={LLM_MODEL})")
print(f"Candidates   : {CANDIDATES_DIR}")
print(f"Scenarios    : {len(FEEDBACK_SCENARIOS)} x {len(FUSIONS)} fusion(s)")
print(f"Retrieval    : top_k_retrieval={TOP_K_RETRIEVAL} top_k_concept={TOP_K_CONCEPT}")


[12:06:43 +   0.0s] Configuration loaded.
Repository   : https://github.com/minhhhduc/bio-linking-.git (main)
KB data      : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/kb.csv
KB artifact  : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/grf_kb.npz
Gold data    : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/translated.jsonl
Output dir   : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/grf
Feedback     : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/grf/grf_feedback_bundle.jsonl (from feedback.ipynb, model=Qwen/Qwen3-32B)
Candidates   : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/grf/candidates
Scenarios    : 5 x 2 fusion(s)
Retrieval    : top_k_retrieval=10 top_k_concept=5


## Stage 1 — Clone or Update the GRF Repository

Colab starts from an empty runtime, so the notebook clones the code repository
and imports the shared modules from `source/common/` and the retrieval modules
from `source/GRF/`. Re-running this cell updates the existing clone instead of
downloading it again. The feedback module is imported for the bundle reader
only — no LLM client is built here.


In [21]:
repo_url = str(require_configured(GIT_REPO_URL, "GIT_REPO_URL"))


def _git(*args: str) -> None:
    log("git " + " ".join(args))
    try:
        subprocess.run(["git", *args], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"git {args[0]} failed with exit code {exc.returncode}.") from exc


if REPO_CLONE_DIR.exists() and not (REPO_CLONE_DIR / ".git").exists():
    raise RuntimeError(
        f"{REPO_CLONE_DIR} exists but is not a git clone. "
        "Remove it or point REPO_CLONE_DIR elsewhere."
    )

if (REPO_CLONE_DIR / ".git").exists():
    log(f"Repository already present at {REPO_CLONE_DIR}; updating branch {GIT_BRANCH!r}.")
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    log(f"Cloning {repo_url} (branch {GIT_BRANCH!r}) into {REPO_CLONE_DIR} ...")
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, repo_url, str(REPO_CLONE_DIR))

SOURCE_DIR = REPO_CLONE_DIR / "source"
if not (SOURCE_DIR / "common" / "__init__.py").is_file():
    raise FileNotFoundError(
        f"{SOURCE_DIR / 'common' / '__init__.py'} is missing; check GIT_BRANCH and the repository layout."
    )
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from common.candidates import load_candidates, save_candidates
from common.data import load_bel_jsonl
from common.encoder import SentenceTransformerEncoder
from common.evaluator import accuracy_at_k
from common.index import SINGLE_FILE_FORMAT, VectorIndex
from common.kb import load_concepts_csv

from GRF.config import PipelineConfig
from GRF.feedback import (
    feedback_namespace,
    load_feedback_outcomes,
    select_feedback_kinds,
)
from GRF.pipeline import EntityLinker

log(f"Repository ready at {REPO_CLONE_DIR}; common and GRF imported from {SOURCE_DIR}.")
log(f"Single-file artifact format: {SINGLE_FILE_FORMAT}")


[12:06:43 +   0.0s] Repository already present at /content/bio_linking; updating branch 'main'.
[12:06:43 +   0.0s] git -C /content/bio_linking fetch --depth 1 origin main
[12:06:44 +   0.3s] git -C /content/bio_linking checkout main
[12:06:44 +   0.3s] git -C /content/bio_linking pull --ff-only origin main
[12:06:44 +   0.4s] Repository ready at /content/bio_linking; common and GRF imported from /content/bio_linking/source.
[12:06:44 +   0.4s] Single-file artifact format: grf-vector-index-npz-v1


## Stage 2 — Mount Google Drive and Locate Inputs

The KB data (`kb.csv`), the KB artifact, the gold data and the feedback bundle
live on Drive, not in the repository. This stage mounts Drive and checks every
input path plus the single candidate output path before any model is loaded.


In [22]:
if MOUNT_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:  # running outside Colab
        raise RuntimeError(
            "Google Colab is required to mount Drive. Run this notebook in Colab, "
            "or set MOUNT_DRIVE = False only if the inputs are already local to the runtime."
        ) from exc
    drive.mount(str(DRIVE_MOUNT_POINT))

RANKING_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

kb_path = Path(require_configured(KB_CSV_PATH, "KB_CSV_PATH"))
artifact_path = Path(require_configured(KB_ARTIFACT_PATH, "KB_ARTIFACT_PATH"))
gold_path = Path(require_configured(GOLD_DATA_PATH, "GOLD_DATA_PATH"))
bundle_path = Path(require_configured(FEEDBACK_BUNDLE_PATH, "FEEDBACK_BUNDLE_PATH"))
summary_path = Path(require_configured(SUMMARY_PATH, "SUMMARY_PATH"))

for label, path in (
    ("KB data", kb_path),
    ("KB artifact", artifact_path),
    ("Gold data", gold_path),
    ("Feedback bundle", bundle_path),
):
    if not path.is_file():
        if label == "Feedback bundle":
            raise FileNotFoundError(
                f"Feedback bundle not found: {path}. Run source/notebooks/GRF/feedback.ipynb first."
            )
        raise FileNotFoundError(f"{label} not found: {path}")
    log(f"{label}: {path} ({path.stat().st_size / 1024 ** 2:.2f} MB)")

if summary_path.suffix.lower() != ".json":
    raise ValueError(f"SUMMARY_PATH must end in .json, got {summary_path.name!r}.")
log(f"Candidate outputs: {CANDIDATES_DIR}")
log(f"Summary output   : {summary_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[12:06:45 +   1.7s] KB data: /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/kb.csv (160.23 MB)
[12:06:45 +   1.7s] KB artifact: /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/grf_kb.npz (9231.94 MB)
[12:06:45 +   1.7s] Gold data: /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/translated.jsonl (0.22 MB)
[12:06:45 +   1.7s] Feedback bundle: /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/grf/grf_feedback_bundle.jsonl (0.70 MB)
[12:06:45 +   1.7s] Candidate outputs: /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/grf/candidates
[12:06:45 +   1.7s] Summary output   : /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/grf/grf_retrieval_summary.json


## Stage 3 — Load the Feedback Bundle and the Gold Data

`load_feedback_outcomes` rebuilds, from the file `feedback.ipynb` wrote, the
request list and the per-mention `(feedbacks, failures)` pairs `link_batch`
takes as `feedback_outcomes`. Those requests ask for **no** feedback
(`feedback_types=()`), which is the whole point: retrieval needs no generator.

The bundle is pinned to `<model>|<prompt digest>`; `feedback_namespace(LLM_MODEL)`
computes the same string from this clone, so a bundle from another model — or
one generated before `prompts.yaml` was edited — raises here instead of being
replayed as if it were current.

The gold data is still loaded, because `save_candidates` and the metrics need
`gold_cui` and `label`, and because the two lists must line up: the bundle is
in gold order, and stage 3 cross-checks every mention, context and token span
against the gold sample before the encoder is loaded.


In [23]:
namespace = feedback_namespace(LLM_MODEL)
log(f"Loading the feedback bundle {bundle_path} (ns={namespace}) ...")
with progress_bar(None, "Loading feedback records", "record") as report:
    requests, feedback_outcomes = load_feedback_outcomes(
        bundle_path, namespace=namespace, progress_callback=report
    )
log(f"Loaded feedback for {len(requests):,} mentions.")
if not requests:
    raise ValueError(f"{bundle_path} contains no records; re-run feedback.ipynb.")

log(f"Loading gold BEL data from {gold_path} ...")
started = time.time()
with progress_bar(None, "Loading gold records", "record") as report:
    loaded = load_bel_jsonl(gold_path, strict=False, progress_callback=report)
samples = list(loaded.samples)
if MAX_SAMPLES is not None:
    if isinstance(MAX_SAMPLES, bool) or not isinstance(MAX_SAMPLES, int) or MAX_SAMPLES < 1:
        raise ValueError("MAX_SAMPLES must be None or a positive integer.")
    samples = samples[:MAX_SAMPLES]

log(
    f"Loaded {len(loaded.samples):,} mention samples "
    f"({len(samples):,} selected) in {time.time() - started:.1f}s."
)
if loaded.issues:
    log(f"WARNING: {len(loaded.issues):,} data issue(s); first: {loaded.issues[0]}")
if not samples:
    raise ValueError(f"{gold_path} produced no usable mention samples.")
if MAX_SAMPLES is not None:
    # A smoke run keeps the two lists aligned: the bundle is in gold order.
    requests = requests[: len(samples)]
    feedback_outcomes = feedback_outcomes[: len(samples)]

if len(samples) != len(requests):
    raise ValueError(
        f"{bundle_path} holds {len(requests):,} feedback records but {gold_path} "
        f"has {len(samples):,} mentions. Re-run feedback.ipynb on the same gold file."
    )

different = [
    (index, sample.mention, request.mention)
    for index, (sample, request) in enumerate(zip(samples, requests))
    if (
        sample.mention,
        sample.context,
        sample.start_token,
        sample.end_token,
    )
    != (
        request.mention,
        request.context,
        request.start_token,
        request.end_token,
    )
]
if different:
    index, gold_mention, bundle_mention = different[0]
    raise ValueError(
        f"{len(different):,} of {len(requests):,} mentions differ between the bundle "
        f"and the gold data; first at record {index}: bundle {bundle_mention!r} vs "
        f"gold {gold_mention!r}. Re-run feedback.ipynb on the same gold file."
    )

with_feedback = sum(1 for feedbacks, _ in feedback_outcomes if feedbacks)
without_feedback = len(feedback_outcomes) - with_feedback
degraded = sum(len(failures) for _, failures in feedback_outcomes)
log(
    f"Bundle matches the gold data record for record: {with_feedback:,} mention(s) "
    f"with feedback, {without_feedback:,} with none, {degraded:,} kind(s) degraded to Q_0."
)


[12:06:45 +   1.7s] Loading the feedback bundle /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/grf/grf_feedback_bundle.jsonl (ns=Qwen/Qwen3-32B|2572e4332b7ac545) ...


Loading feedback records: 0record [00:00, ?record/s]

[12:06:45 +   1.7s] Loaded feedback for 959 mentions.
[12:06:45 +   1.7s] Loading gold BEL data from /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/translated.jsonl ...


Loading gold records: 0record [00:00, ?record/s]

[12:06:45 +   1.7s] Loaded 959 mention samples (959 selected) in 0.0s.
[12:06:45 +   1.7s] Bundle matches the gold data record for record: 959 mention(s) with feedback, 0 with none, 0 kind(s) degraded to Q_0.


## Stage 4 — Load the KB Artifact and the Encoder

`VectorIndex.load_npz` restores the vectors and metadata from the single file.
The encoder model itself is not stored in the artifact, so this stage loads the
same checkpoint recorded in `encoder_info` and checks the dimensions agree.

The GPU copy of the vectors is **not** taken here. `index.use_gpu()` routes
`pycauset.matmul` to the GPU, and it is enabled in stage 7, right before the
scan. The KB itself stays memory-mapped on disk, so nothing else competes for
that device memory.


In [24]:
log(f"Loading KB artifact {artifact_path} ...")
# The .npz member vectors.npy is streamed to local disk and imported as a
# file-backed pycauset matrix, so the KB never sits in one RAM array.
index = VectorIndex.load_npz(artifact_path, cache_dir=Path("/content/grf_kb_npy"))
log(f"Artifact loaded: {len(index):,} entries x {index.vectors.shape[1]} dims.")

stored_encoder = (index.encoder_info or {}).get("model")
if stored_encoder is None:
    log("WARNING: artifact has no encoder_info; assuming ENCODER_MODEL is the same checkpoint.")
elif stored_encoder != ENCODER_MODEL:
    raise ValueError(
        f"Artifact was built with encoder {stored_encoder!r}, but ENCODER_MODEL is "
        f"{ENCODER_MODEL!r}. Set ENCODER_MODEL to the encoder used by build_KB.ipynb."
    )

if DEVICE == "cuda":
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError(
            "DEVICE='cuda' but no CUDA device is available. In Colab: "
            "Runtime > Change runtime type > Hardware accelerator > GPU."
        )
    log(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    log("Running on CPU (DEVICE='cpu').")

log(f"Loading encoder {ENCODER_MODEL!r} on {DEVICE} ...")
encoder = SentenceTransformerEncoder(
    ENCODER_MODEL, batch_size=ENCODER_BATCH_SIZE, device=DEVICE
)
if encoder.dim != index.vectors.shape[1]:
    raise ValueError(
        f"Encoder dimension {encoder.dim} does not match artifact dimension "
        f"{index.vectors.shape[1]}."
    )
log(f"Encoder ready: dim={encoder.dim}.")


[12:06:45 +   1.7s] Loading KB artifact /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/grf_kb.npz ...
[12:10:14 + 210.3s] Artifact loaded: 3,342,067 entries x 768 dims.
[12:10:14 + 210.3s] GPU: Tesla T4
[12:10:14 + 210.3s] Loading encoder 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext' on cuda ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[12:10:16 + 212.6s] Encoder ready: dim=768.


/content/bio_linking/source/common/encoder.py:68: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  declared = self._model.get_sentence_embedding_dimension()


## Stage 5 — Load the KB Data from Google Drive

The KB CSV on Drive is the authoritative source of preferred concept names
used by the LLM re-ranker in `reranking.ipynb`. It is loaded here and cross-checked
against the CUIs stored in the artifact so a stale `grf_kb.npz` fails loudly
instead of mixing an old index with a new KB.


In [25]:
log(f"Loading KB data from {kb_path} ...")
started = time.time()
with progress_bar(None, "Parsing KB records", "record") as report:
    kb_concepts = load_concepts_csv(kb_path, progress_callback=report)
log(f"Loaded {len(kb_concepts):,} KB concepts in {time.time() - started:.1f}s.")
if not kb_concepts:
    raise ValueError(f"{kb_path} produced no KB concepts.")

concept_names = {concept.cui: concept.name for concept in kb_concepts}
artifact_cuis = set(index.cuis)
kb_cuis = set(concept_names)

missing = artifact_cuis - kb_cuis
if missing:
    raise ValueError(
        f"{len(missing):,} CUIs in the KB artifact are missing from {kb_path}; "
        "the artifact and the KB data do not match. Rebuild it with build_KB.ipynb."
    )
extra = kb_cuis - artifact_cuis
if extra:
    log(
        f"WARNING: {len(extra):,} KB CUIs are absent from the artifact "
        "(the artifact may be stale or filtered)."
    )
log(f"Concept names ready: {len(concept_names):,} CUIs from the KB data.")

[12:10:16 + 212.6s] Loading KB data from /content/drive/MyDrive/work/nlp/UMLS/entity_linking/data/kb.csv ...


Parsing KB records: 0record [00:00, ?record/s]

[12:10:48 + 244.9s] Loaded 3,342,067 KB concepts in 32.3s.
[12:10:52 + 248.8s] Concept names ready: 3,342,067 CUIs from the KB data.


## Stage 6 — Validate the Scenario Matrix

`PipelineConfig` is validated on construction (fusion kind, `n_syn` in
{1, 3, 5, 10}, `top_k_concept` in 1..10). Each scenario below builds its own
config and its own linker with **no** `feedback_generator`: every request
carries its feedback already, so a request that did ask for feedback would fail
rather than silently reach a missing model.

`select_feedback_kinds` keeps only the scenario's kinds from each mention's
feedbacks and failures, matched on the record's `kind` key — so one all-kind
bundle serves the single-type and All-in-One scenarios. Every requested kind
must be present for every mention (as feedback or as a recorded failure); a
missing kind stops the run instead of silently fusing fewer components.

No re-ranking happens in this notebook: `rerank="top1"` only fills
`Response.cui` with the ranking's first entry. The saved candidates are
re-ranked separately by `reranking.ipynb`.

Re-run this cell after changing any constant or re-running an upstream cell:
stage 7 rebuilds everything from `scenarios` and `feedback_outcomes`.


In [26]:
# Every scenario reads the same bundle; validate once that it holds every kind
# any scenario asks for, so a missing kind fails here and not mid-matrix.
all_kinds = tuple(dict.fromkeys(kind for _, kinds in FEEDBACK_SCENARIOS for kind in kinds))
select_feedback_kinds(feedback_outcomes, all_kinds)

scenarios = [
    (f"{name}_{fusion}", kinds, fusion)
    for name, kinds in FEEDBACK_SCENARIOS
    for fusion in FUSIONS
]
acc_ks = tuple(sorted({min(k, TOP_K_CONCEPT) for k in ACC_K}))
log(
    f"Retrieval pipeline ready: {len(index):,} index entries, "
    f"{len(concept_names):,} concepts; {len(scenarios)} scenario(s) from "
    f"{bundle_path.name}; n_syn={N_SYN} acc@{list(acc_ks)}."
)


[12:10:53 + 249.3s] Retrieval pipeline ready: 3,342,067 index entries, 3,342,067 concepts; 10 scenario(s) from grf_feedback_bundle.jsonl; n_syn=5 acc@[1, 5].


## Stage 7 - Run Every Scenario (Retrieval, Save, acc@k)

`link_batch(..., feedback_outcomes=...)` replays the bundle and runs the
retrieval phases. Every request's `Q_0` is the **mean of its mention's own
token states inside its sentence** (`encode_mention_in_context`) - no `E(m)`
term added - or `encoder.encode_raw(mention)` when `USE_CONTEXT_QUERY = False`,
all in **one forward pass**; every feedback text of the batch is then embedded
in one call. Pipeline 1 (`fusion="vector"`) fuses them in a single contraction
and scans once with `R = Q_GRF @ A.T`; pipeline 2 (`fusion="rrf"`) forms no
fused vector and instead scans **once per query component** - `Q_0`, then
`Q_F1`, ... , `Q_Fn` - each scan a `R = Q_x @ A.T` matrix for the whole batch, so
a mention with `n` feedback gets `n + 1` ranked lists. Either way top-k runs
along each row in NumPy, on the block `pycauset.matmul` returned.

The loop below walks every `(feedback subset, fusion)` pair: it builds a
`PipelineConfig`, retrieves, saves the candidates to
`CANDIDATES_DIR/grf_<scenario>_n<n>_k<k>_candidates.jsonl`, reloads that file to
prove it is usable, and records `acc@k` - the share of mentions whose gold CUI
is among the first `k` saved candidates. Each scenario's bars are prefixed with
its name.

CUDA routing is enabled here, then the scan runs. The KB stays memory-mapped;
the feedback model lives in the other notebook.

This notebook **stops at the ranking**: `rerank="top1"` only copies the
ranking's first entry into `Response.cui`, so no re-ranking work happens here.
The saved candidates are re-ranked later, by `reranking.ipynb`.


In [27]:
if DEVICE == "cuda":
    try:
        index.use_gpu(DEVICE)
    except RuntimeError as exc:
        log(f"WARNING: GPU search unavailable ({exc}); using the CPU scan.")
    else:
        log(f"Search device: {index.search_device} (exact cosine on GPU).")

# One bundle, every row of the paper's matrix: retrieve, save, verify and score
# each scenario before moving on, so only one response list is live at a time.
results = []
for slug, kinds, fusion in scenarios:
    config = PipelineConfig(
        encoder=ENCODER_MODEL,
        feedback_types=kinds,
        n_syn=N_SYN,
        target_kb=TARGET_KB,
        top_k_retrieval=TOP_K_RETRIEVAL,
        top_k_concept=TOP_K_CONCEPT,
        fusion=fusion,
        rerank="top1",              # no re-rank here; reranking.ipynb does it
        llm_model=LLM_MODEL,        # provenance: the model that wrote the bundle
        index_kb=INDEX_KB,
    )
    linker = EntityLinker(
        index,
        encoder,
        config,
        feedback_generator=None,    # the feedback is in the bundle; no LLM here
        concept_names=concept_names,
        use_context_query=USE_CONTEXT_QUERY,
    )
    with phase_progress(f"{slug}: ") as report:
        started = time.time()
        responses = linker.link_batch(
            requests,
            feedback_outcomes=select_feedback_kinds(feedback_outcomes, kinds),
            progress_callback=report,
        )
    empty = sum(1 for response in responses if not response.candidates)
    log(
        f"{slug}: candidates for {len(responses):,} mentions in "
        f"{time.time() - started:.1f}s"
        + (f"; WARNING: {empty:,} mention(s) with no candidates" if empty else "")
    )

    candidates_path = (
        CANDIDATES_DIR / f"grf_{slug}_n{N_SYN}_k{TOP_K_CONCEPT}_candidates.jsonl"
    )
    with progress_bar(len(samples), f"{slug}: saving", "record") as report:
        save_candidates(candidates_path, samples, responses, progress_callback=report)
    reloaded_samples, reloaded_responses = load_candidates(candidates_path)
    assert len(reloaded_samples) == len(samples), (slug, len(reloaded_samples), len(samples))
    assert len(reloaded_responses) == len(responses), (slug, len(reloaded_responses))
    assert [sample.gold_cui for sample in reloaded_samples] == [
        sample.gold_cui for sample in samples
    ], slug
    assert [response.candidates for response in reloaded_responses] == [
        response.candidates for response in responses
    ], slug
    assert [
        response.debug.get("concept_ranking") or [] for response in reloaded_responses
    ] == [
        response.debug.get("concept_ranking") or [] for response in responses
    ], slug

    row = {
        "scenario": slug,
        "feedback_types": list(kinds),
        "fusion": fusion,
        "candidates_file": str(candidates_path),
        "n": len(reloaded_samples),
        "acc@k": {},
        "by_label": {},
    }
    for k in acc_ks:
        overall, by_label = accuracy_at_k(reloaded_samples, reloaded_responses, k)
        row["acc@k"][str(k)] = overall
        row["by_label"][str(k)] = {
            str(label): value for label, value in by_label.items()
        }
    results.append(row)
log(f"Retrieved and saved {len(results)} scenario(s).")


[12:10:55 + 251.5s] Search device: cuda (exact cosine on GPU).


none_vector: encode Q0: 0item [00:00, ?item/s]

none_vector: encode feedback: 0item [00:00, ?item/s]

none_vector: retrieval:   0%|          | 0/1 [00:00<?, ?item/s]

none_vector: aggregation:   0%|          | 0/959 [00:00<?, ?item/s]

none_vector: ranking:   0%|          | 0/959 [00:00<?, ?item/s]

... (62 outputs left out)

all_rrf: ranking:   0%|          | 0/959 [00:00<?, ?item/s]

[12:14:39 + 475.9s] all_rrf: candidates for 959 mentions in 48.2s


all_rrf: saving:   0%|          | 0/959 [00:00<?, ?record/s]

[12:14:40 + 476.2s] Retrieved and saved 10 scenario(s).


## Stage 8 - Summary Table and JSON

One row per scenario with `acc@1`/`acc@5`, then the same numbers (plus
per-label breakdowns) are written to `SUMMARY_PATH` and read back to verify the
file. With one gold CUI per mention, `acc@k` counts a hit exactly when the gold
concept is among the first `k` candidates.


In [28]:
ks = [int(k) for k in results[0]["acc@k"]] if results else []
header = f"{'scenario':<22}{'n':>7}" + "".join(f"{f'acc@{k} %':>10}" for k in ks)
print(header)
for row in results:
    line = f"{row['scenario']:<22}{row['n']:>7}"
    line += "".join(f"{100 * row['acc@k'][str(k)]:>10.2f}" for k in ks)
    print(line)

payload = {
    "metric": "acc@k",
    "encoder": ENCODER_MODEL,
    "n_syn": N_SYN,
    "target_kb": TARGET_KB,
    "top_k_retrieval": TOP_K_RETRIEVAL,
    "top_k_concept": TOP_K_CONCEPT,
    "use_context_query": USE_CONTEXT_QUERY,
    "llm_model": LLM_MODEL,
    "bundle": str(bundle_path),
    "n_samples": len(samples),
    "scenarios": results,
}
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
saved = json.loads(summary_path.read_text(encoding="utf-8"))
assert saved["scenarios"] == results
log(f"Saved and verified {summary_path} ({len(results)} scenario(s)).")
log("Candidate retrieval complete. Next: source/notebooks/GRF/reranking.ipynb")


scenario                    n   acc@1 %   acc@5 %
none_vector               959     60.58     79.46
none_rrf                  959     60.58     79.46
def_vector                959     59.54     75.91
def_rrf                   959     53.08     75.29
syn_vector                959     64.75     83.52
syn_rrf                   959     63.09     84.67
std_vector                959     63.09     84.25
std_rrf                   959     62.04     83.73
all_vector                959     63.30     83.42
all_rrf                   959     63.40     84.05
[12:14:40 + 476.3s] Saved and verified /content/drive/MyDrive/work/nlp/UMLS/entity_linking/output/grf/grf_retrieval_summary.json (10 scenario(s)).
[12:14:40 + 476.3s] Candidate retrieval complete. Next: source/notebooks/GRF/reranking.ipynb


## Stage 9 — Release Memory

Candidates are saved and the metrics are printed, so release both GPU
residents: the index's GPU copy (`release_gpu`) and the encoder. Then drop the
KB concepts, the request list, the feedback outcomes and the per-scenario
leftovers, and empty the CUDA cache. `samples` stays for reference.
Re-run stages 4–9 if you need the live pipeline again.


In [29]:
if "index" in globals():
    index.release_gpu()

for name in (
    "linker",
    "index",
    "encoder",
    "kb_concepts",
    "concept_names",
    "requests",
    "feedback_outcomes",
    "results",
    "responses",
    "reloaded_samples",
    "reloaded_responses",
):
    globals().pop(name, None)
gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_b, total_b = torch.cuda.mem_get_info()
        log(
            f"CUDA cache emptied: {free_b / 1024 ** 3:.2f}/"
            f"{total_b / 1024 ** 3:.2f} GiB free."
        )
except ImportError:
    pass

log("Retrieval memory released (index, encoder, KB names, requests, feedback).")


[12:14:41 + 477.9s] CUDA cache emptied: 14.41/14.56 GiB free.
[12:14:41 + 477.9s] Retrieval memory released (index, encoder, KB names, requests, feedback).
